In [ ]:
!pip -q install pandas scikit-learn transformers torch openpyxl google-api-python-client

In [ ]:
# =====================================================================
#  ONE-CELL AUDIT RUNNER  (real data, 389 audit-eligible)
#  Paste your fresh Perspective key below, run, pick the REAL file.
# =====================================================================
PERSPECTIVE_API_KEY = 'PASTE_YOUR_NEW_KEY_HERE'   # <-- your fresh key

import time, pandas as pd, numpy as np, torch
from tqdm.auto import tqdm
from google.colab import files
from googleapiclient import discovery
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ---- 1. Upload the REAL audit-ready file ----
print("\nUpload  ethiopia_moderation_dataset_REAL_audit_ready.xlsx  when prompted:")
up = files.upload()
DATASET_FILE = next(iter(up.keys()))
df_all = pd.read_excel(DATASET_FILE, sheet_name='Dataset')
df = df_all[df_all['IncludeInAudit'] == True].copy().reset_index(drop=True)
print(f"\n>>> Audit-eligible rows: {len(df)}   (must be 389)")
print("    by language:", df['Language'].value_counts().to_dict())
assert len(df) == 389, "NOT the real file — stop and upload the REAL audit-ready file."

# ---- 2. Perspective API: native mode + forced-English ----
client = discovery.build('commentanalyzer','v1alpha1',
    developerKey=PERSPECTIVE_API_KEY,
    discoveryServiceUrl='https://commentanalyzer.googleapis.com/$discovery/rest?version=v1alpha1',
    static_discovery=False)

def persp(text, languages):
    req={'comment':{'text':str(text)[:20000]},'requestedAttributes':{'TOXICITY':{}},
         'doNotStore':True}
    if languages is not None: req['languages']=languages
    try:
        r=client.comments().analyze(body=req).execute()
        return r['attributeScores']['TOXICITY']['summaryScore']['value'],'ok'
    except Exception as e:
        return None, type(e).__name__

lang_map={'Amharic':['am'],'Afan Oromo':None}  # native: am for Amharic, auto-detect for Oromo
nat_s,nat_st,fen_s,fen_st=[],[],[],[]
for _,row in tqdm(df.iterrows(),total=len(df),desc='Perspective'):
    s,st=persp(row['OriginalText'], lang_map.get(row['Language']))      # native
    nat_s.append(s); nat_st.append(st)
    s2,st2=persp(row['OriginalText'], ['en'])                           # forced-English
    fen_s.append(s2); fen_st.append(st2)
    time.sleep(1.05)
df['persp_native_score']=nat_s; df['persp_native_status']=nat_st
df['persp_native_pred_hate']=[(s is not None and s>=0.5) for s in nat_s]
df['persp_forced_en_score']=fen_s; df['persp_forced_en_status']=fen_st
df['persp_forced_en_pred_hate']=[(s is not None and s>=0.5) for s in fen_s]
print("Native statuses:", pd.Series(nat_st).value_counts().to_dict())
print("Forced-EN statuses:", pd.Series(fen_st).value_counts().to_dict())

# ---- 3. AfriHate (AfroXLMR) ----
AFRI='Davlan/afro-xlmr-base-hate-v1'
tok=AutoTokenizer.from_pretrained(AFRI)
mdl=AutoModelForSequenceClassification.from_pretrained(AFRI).to(DEVICE).eval()
print('AfriHate id2label:', mdl.config.id2label)
@torch.no_grad()
def afri(texts,bs=32):
    out=[]
    for i in tqdm(range(0,len(texts),bs),desc='AfriHate'):
        b=[str(t)[:512] for t in texts[i:i+bs]]
        enc=tok(b,padding=True,truncation=True,max_length=256,return_tensors='pt').to(DEVICE)
        for p in mdl(**enc).logits.argmax(-1).cpu().numpy():
            out.append(mdl.config.id2label[int(p)])
    return out
df['afrihate_label']=afri(df['OriginalText'].tolist())
df['afrihate_pred_hate']=df['afrihate_label'].map(lambda l: any(k in str(l).lower() for k in ['hate','abus']))

# ---- 4. Amharic mBERT (Amharic rows only) ----
MB='amengemeda/amharic-hate-speech-detection-mBERT'
tmb=AutoTokenizer.from_pretrained(MB)
mmb=AutoModelForSequenceClassification.from_pretrained(MB).to(DEVICE).eval()
print('mBERT id2label:', mmb.config.id2label)
amh=df['Language']=='Amharic'
@torch.no_grad()
def mbert(texts,bs=32):
    out=[]
    for i in tqdm(range(0,len(texts),bs),desc='mBERT'):
        b=[str(t)[:512] for t in texts[i:i+bs]]
        enc=tmb(b,padding=True,truncation=True,max_length=256,return_tensors='pt').to(DEVICE)
        for p in mmb(**enc).logits.argmax(-1).cpu().numpy():
            out.append(int(p))
    return out
df['mbert_pred_hate']=None
df.loc[amh,'mbert_pred_hate']=[bool(p==1) for p in mbert(df.loc[amh,'OriginalText'].tolist())]

# ---- 5. Save predictions (includes gold + language so I can evaluate) ----
cols=['PostID','Language','FinalLabel','Label3Class','GoldHate',
      'persp_native_score','persp_native_status','persp_native_pred_hate',
      'persp_forced_en_score','persp_forced_en_status','persp_forced_en_pred_hate',
      'afrihate_label','afrihate_pred_hate','mbert_pred_hate']
OUT='predictions_real389.xlsx'
df[cols].to_excel(OUT,index=False)
print('\nSaved:',OUT)
files.download(OUT)
print("\nDONE. Send predictions_real389.xlsx back to Claude for evaluation.")

Device: cpu

Upload  ethiopia_moderation_dataset_REAL_audit_ready.xlsx  when prompted:
